In [1]:
import sys
import os

# Adjust the path to where the src folder is located
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

In [2]:
import pandas as pd
import torch
from language_models.dictionary_corpus import Dictionary
from language_models.model import RNNModel
import torch.nn.functional as F
import math
import numpy as np

In [3]:
test = pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/eval_surprisal/sentences/items_RelativeClause.pivot.csv')

In [4]:
test

,Unnamed: 0,item,condition,targetPosition,Sentence,Unnamed: 4,Option1,Option0,Ans,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,0,25,RC_Subj,5,The bus driver who followed the kids wondered ...,Who was being followed?,The bus driver,The kids,0,RC-Gap,1.0,NaN,NaN,NaN,NaN,NaN
1,1,25,RC_Obj,7,The bus driver who the kids followed wondered ...,Who was being followed?,The bus driver,The kids,1,RC-Gap,NaN,NaN,NaN,1.0,NaN,NaN
2,2,26,RC_Subj,4,The chef who distracted the cameraman poured t...,Who was being distracted?,The chef,The cameraman,0,RC-Gap,1.0,NaN,NaN,NaN,NaN,NaN
3,3,26,RC_Obj,6,The chef who the cameraman distracted poured t...,Who was being distracted?,The chef,The cameraman,1,RC-Gap,NaN,NaN,NaN,1.0,NaN,NaN
4,4,27,RC_Subj,4,The children who woke the father bothered him ...,Who was woken up?,The children,The father,0,RC-Gap,NaN,NaN,NaN,1.0,NaN,NaN
5,5,27,RC_Obj,6,The children who the father woke bothered him ...,Who was woken up?,The children,The father,1,RC-Gap,1.0,NaN,NaN,NaN,NaN,NaN
6,6,28,RC_Subj,4,The class that disliked the teacher skimmed th...,Who was disliked?,The class,The teacher,0,RC-Gap,NaN,NaN,NaN,1.0,NaN,NaN
7,7,28,RC_Obj,6,The class that the teacher disliked skimmed th...,Who was disliked?,The class,The teacher,1,RC-Gap,1.0,NaN,NaN,NaN,NaN,NaN
8,8,29,RC_Subj,4,The dancer that loved the audience ignored som...,Who was loved?,The dancer,The audience,0,RC-Gap,NaN,1.0,NaN,NaN,NaN,NaN
9,9,29,RC_Obj,6,The dancer that the audience loved ignored som...,Who was loved?,The dancer,The audience,1,RC-Gap,NaN,NaN,NaN,NaN,1.0,NaN


In [5]:
train = pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/eval_surprisal/sentences/items_filler.pivot.csv')

In [6]:
train

,Unnamed: 0,item#_in_Provo,item,Sentence,Question,Option1,Option0,Correct answer
0,0,1.0,73,There are now rumblings that Apple might soon ...,Which market is Apple rumored to enter?,Smart watches,Self-driving cars,1
1,1,28.0,74,A bill was drafted and introduced into Parliam...,Who hates the bill most?,Construction workers,Farmers,0
2,2,8.0,75,The human body can tolerate only a small range...,At which point are people resistent to a wider...,When sleeping,When exercising,0
3,3,6.0,76,Seeing Peter slowly advancing upon him through...,Did the man escape Peter\'s attempt to attack?,Yes,No,1
4,4,12.0,77,"Some months later, Michael Larson saw another ...",Did Larson see more than one opportunity?,Yes,No,1
5,5,13.0,78,"Bob Murphy, the Senior PGA Tour money leader w...",Is the leader cocnered about the weather?,Yes,No,0
6,6,15.0,79,"Greg Anderson, considered a key witness by the...",Was Greg Anderson asked to testify?,Yes,No,1
7,7,17.0,80,Owls are more flexible than humans because a b...,Are humans less flexible than owls?,Yes,No,1
8,8,18.0,81,"Even in the same animal, not all bites are the...","For a given animal, can you expect that some b...",Yes,No,1
9,9,20.0,82,"Buck did not like it, but he bore up well to t...",How did Buck deal with his job?,He quit,He carried on,0


In [7]:
train = train['Sentence']
test = test['Sentence']

In [8]:
df = pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/eval_surprisal/sentences/Fillers.csv')

In [9]:
df.columns

Index(['Time', 'MD5', 'Type', 'WordPosition', 'EachWord', 'EventTime',
       'Sentence', 'Question', 'Answer', 'List', 'item', 'RT_Answering',
       'AccLow', 'nonEng', 'CriticalPosition', 'consec', 'CONSTRUCTION',
       'correct', 'ROI', 'RT', 'AMBIG', 'AMBUAMB', 'RTacross3words',
       'trialnumber'],
      dtype='object')

## 1. Gather surprisal per sentences to create train and test sets for linking model

1. tokenize dataset (word level)
2. process it 
3. gather surprisal

In [10]:


# --- Settings ---
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
model_path = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/epoch_40.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_csv = "test.csv"
train = pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/eval_surprisal/sentences/items_filler.pivot.csv')
train = train['Sentence']
# --- Sentences to process ---
sentences = train

# --- Load vocab ---
dictionary = Dictionary(data_path)

# --- Tokenizer ---
def tokenize(sent):
    sent = sent.strip()
    if sent == "": return []
    sent = " ,".join(sent.split(","))
    if sent[-1] in [".", "?", "!"]:
        sent = sent[:-1] + " " + sent[-1]
    sent = " 's".join(sent.split("'s"))
    sent = " n't".join(sent.split("n't"))
    return sent.split()

# --- Load model ---
model = RNNModel(
    rnn_type="LSTM", 
    ntoken=len(dictionary), 
    ninp=650, 
    nhid=650, 
    nlayers=2, 
    dropout=0.2, 
    tie_weights=False
).to(device)

state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict['model_state_dict'])
model.eval()
results = []
# --- Process sentences ---
for sent_idx, sentence in enumerate(sentences):
    tokens = ["<eos>"] + tokenize(sentence)
    indexed = [dictionary.word2idx.get(w, dictionary.word2idx["<unk>"]) for w in tokens]
    input_tensor = torch.tensor(indexed, dtype=torch.long).view(-1, 1).to(device)

    with torch.no_grad():
        hidden = model.init_hidden(1)
        output, _ = model(input_tensor[:-1], hidden)
        target = input_tensor[1:].squeeze(1)
        log_probs = F.log_softmax(output.squeeze(1), dim=-1)

        print(f"\nSentence {sent_idx+1}: {sentence}")
        print(f"{'Token':<15}{'Surprisal (bits)':>20}")
        for i in range(len(target)):
            token = tokens[i+1]  # skip <eos>
            surprisal = -log_probs[i, target[i]].item() / np.log(2)
            results.append({
                "sentence_index": sent_idx,
                "sentence": sentence,
                "token_position": i,
                "token": token,
                "surprisal_bits": surprisal
            })

# --- Save to CSV ---
df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)
print(f"\n✅ Saved surprisal data to: {output_csv}")



Sentence 1: There are now rumblings that Apple might soon invade the smart watch space, though the company is maintaining its customary silence.
Token              Surprisal (bits)

Sentence 2: A bill was drafted and introduced into Parliament several times but met with great opposition, mostly from farmers.
Token              Surprisal (bits)

Sentence 3: The human body can tolerate only a small range of temperature, especially when the person is engaged in vigorous activity.
Token              Surprisal (bits)

Sentence 4: Seeing Peter slowly advancing upon him through the air with dagger poised, he sprang upon the bulwarks to cast himself into the sea.
Token              Surprisal (bits)

Sentence 5: Some months later, Michael Larson saw another opportunity to stack the odds in his favor with a dash of ingenuity.
Token              Surprisal (bits)

Sentence 6: Bob Murphy, the Senior PGA Tour money leader with seven hundred thousand, says heat shouldn't be a factor.
Token          

In [ ]:

# --- Settings ---
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
model_path = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/epoch_40.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_csv = "filtered_surprisal.csv"
spillover = 2  # Number of tokens after critical position to keep

# --- Load data ---
df = pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/eval_surprisal/sentences/items_filler.pivot.csv')
sentences = df['Sentence'].tolist()
#critical_positions = df['disambPosition_0idx'].tolist()

# --- Load vocab ---
dictionary = Dictionary(data_path)

# --- Tokenizer ---
def tokenize(sent):
    sent = sent.strip()
    if sent == "": return []
    sent = " ,".join(sent.split(","))
    if sent[-1] in [".", "?", "!"]:
        sent = sent[:-1] + " " + sent[-1]
    sent = " 's".join(sent.split("'s"))
    sent = " n't".join(sent.split("n't"))
    return sent.split()

# --- Load model ---
model = RNNModel(
    rnn_type="LSTM", 
    ntoken=len(dictionary), 
    ninp=650, 
    nhid=650, 
    nlayers=2, 
    dropout=0.2, 
    tie_weights=False
).to(device)

state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict['model_state_dict'])
model.eval()

results = []

# --- Process sentences ---
for sent_idx, (sentence, crit_idx) in enumerate(zip(sentences, critical_positions)):
    tokens = ["<eos>"] + tokenize(sentence)
    indexed = [dictionary.word2idx.get(w, dictionary.word2idx["<unk>"]) for w in tokens]
    input_tensor = torch.tensor(indexed, dtype=torch.long).view(-1, 1).to(device)

    with torch.no_grad():
        hidden = model.init_hidden(1)
        output, _ = model(input_tensor[:-1], hidden)
        target = input_tensor[1:].squeeze(1)
        log_probs = F.log_softmax(output.squeeze(1), dim=-1)

        for i in range(len(target)):
            if i < crit_idx or i > crit_idx + spillover:
                continue  # Skip if not in critical+spillover window

            token = tokens[i+1]  # skip <eos>
            surprisal = -log_probs[i, target[i]].item() / np.log(2)
            results.append({
                "sentence_index": sent_idx,
                "sentence": sentence,
                "token_position": i,
                "token": token,
                "surprisal_bits": surprisal
            })

# --- Save to CSV ---
df_out = pd.DataFrame(results)
df_out.to_csv(output_csv, index=False)
print(f"\n✅ Saved filtered surprisal data to: {output_csv}")


KeyError: 'disambPosition_0idx'

## Steps : 
